# k8a — Câu hỏi tổng hợp cho fine-tune reranker (GPU T4 ×2, vLLM)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On. **Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`.
Lấy ngẫu nhiên đoạn nguồn trong kho (`N`: vi/zh/en — có PubMed vì đáp án private có), teacher `MODEL` (Qwen3-32B-AWQ,
chỉ dùng lúc huấn luyện) viết mỗi đoạn một câu hỏi tiếng Việt kiểu người dùng + bản dịch en/zh. KHÔNG dùng câu thi.
**Chạy thử trước:** `LIMIT = 100` → đọc `samples.md` + tốc độ (token/s) ở log, rồi `LIMIT = 0`.
**v2 (30/10, mặc định):** `VARIANT = 'synth2'`, `STYLE = 'mix'` (câu ngắn / vừa / dài theo phân bố độ dài câu thi),
`QA_CAP = 0.3` (trang hỏi-đáp tối đa 30% mỗi ngôn ngữ). Lý do: Qwen-ft v1 lệch nhiều nhất ở câu dài và đẩy trang
hỏi-đáp tiếng Trung lên. v1: `VARIANT = 'synth'`, `STYLE = 'none'`, `QA_CAP = 1.0`.
**Đầu ra:** `<VARIANT>/source.parquet`, `<VARIANT>/gen.jsonl`, `qexp/<VARIANT>/queries_{vi,en,zh}.parquet` → *New
Dataset* **`mir2-synth2`** → k7 (`MODES = ['synth2']`, `FUSED = ['synth2_lin07']`, `RERANK = False`) → k8b.
**Đợt bổ sung:** Add Input output k8a cũ cùng VARIANT, `TAG = 'pm'`, `N = 'en:2000'` → chỉ sinh phần mới, `qexp/<VARIANT>`
dựng lại từ MỌI đợt.
**vLLM trên T4:** `VLLM_SPEC` ghim bản dùng PyTorch CUDA 12.x như Kaggle (bản vLLM mới nhất kéo PyTorch CUDA
13 → lệch torchaudio của Kaggle và cần driver ≥ 580). Ô cài đặt gỡ torchaudio (không dùng) và dừng nếu torch mới không
thấy GPU. Lỗi khi nạp model → thử `VLLM_SPEC = 'vllm==0.11.0'`, hoặc `MODEL = 'Qwen/Qwen3-14B-AWQ'` (nhanh ~2 lần, chất
lượng thấp hơn).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k8a-v5 (15/10/2026)'
VARIANT = 'synth2'                   # 'synth' = v1 (27/10); 'synth2' = v2
N = 'vi:2400,zh:1400,en:2200'        # dư vì bị bỏ qua (v1 hợp lệ: vi ~51%, zh ~74%; en xem log)
STYLE = 'mix'                        # 'mix' = ngắn 25% / vừa 50% / dài 25%; 'none' = như v1
QA_CAP = 0.3                         # tỉ lệ tối đa trang hỏi-đáp mỗi ngôn ngữ; 1.0 = như v1
TAG = ''                             # '' = đợt đầu; 'pm' = đợt bổ sung (Add Input output k8a cũ)
MODEL = 'Qwen/Qwen3-32B-AWQ'
LIMIT = 100                          # >0: chỉ N đoạn đầu (chạy thử); 0 = tất cả
VLLM_SPEC = 'vllm==0.9.2'            # PyTorch 2.7 (CUDA 12.x), T4 chạy engine V0
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
def _ver(h):                                                  # output notebook cũ (vd mir2-chunks/code) cũng có mirret/
    v = os.path.join(os.path.dirname(os.path.dirname(h)), 'VERSION.txt')
    return open(v, encoding='utf-8').read().strip() if os.path.exists(v) else ''
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True),
              key=lambda h: (_ver(h), -len(h)), reverse=True)   # bản mã mới nhất theo VERSION.txt (ngày giờ)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks',))

In [ ]:
# (2) vLLM + đoạn nguồn (CPU); đợt bổ sung: chép các đợt trước từ Input
sh('nvidia-smi --query-gpu=name,driver_version --format=csv,noheader')
sh(f'python -m pip install -q "{VLLM_SPEC}"', 'pip_vllm')
sh('python -m pip uninstall -y -q torchaudio')       # vLLM thay torch của Kaggle; torchaudio cũ lệch CUDA, không dùng tới
sh('python -c "import torch, vllm; print(vllm.__version__, torch.__version__, torch.version.cuda, torch.cuda.is_available())"')
try:
    sh('python -c "import sys, torch; sys.exit(0 if torch.cuda.is_available() else 3)"')
except Exception:
    raise SystemExit('torch mới không thấy GPU (driver Kaggle không hỗ trợ CUDA của bản torch này) → đổi VLLM_SPEC')
for d in (locate_all(f'{VARIANT}/gen.jsonl') if TAG else []):
    shutil.copytree(f'{d}/{VARIANT}', f'{K.work}/{VARIANT}', dirs_exist_ok=True)
    print('đợt trước:', d, sorted(os.listdir(f'{d}/{VARIANT}')))
assert not TAG or os.path.exists(f'{K.work}/{VARIANT}/gen.jsonl'), 'TAG đợt bổ sung: Add Input output k8a đợt đầu'
VA = f' --variant {VARIANT}'
TG = f' --tag {TAG}' if TAG else ''
sh(f'python -m mirret.synth sample --config {K.cfg} --n {N} --qa-cap {QA_CAP}{TG}{VA}', 'sample')

In [ ]:
# (3) Sinh câu hỏi (2 GPU, tensor parallel) → câu hỏi cho k7 (gộp mọi đợt)
lim = f' --limit {LIMIT}' if LIMIT else ''
sh(f'python -m mirret.synth generate --config {K.cfg} --model {MODEL} --tp 2 --style {STYLE}{lim}{TG}{VA}', 'generate')
sh(f'python -m mirret.synth build --config {K.cfg}{VA}', 'build')
print(open(f'{K.work}/qexp/{VARIANT}/samples.md', encoding='utf-8').read())
report('k8a')